# Manual MLPs for Classification and Regression

**Hayden Willen** | DS 4021 Machine Learning II | University of Virginia School of Data Science

This project builds multi-layer perceptrons from scratch in PyTorch for three tasks, which are binary classification, multi-class classification and regression. Each model sets up its own weights, runs its own forward pass, computes its own cost and updates its weights with mini-batch stochastic gradient descent. No `torch.nn` layers or optimizers are used.

Each model is trained under 16 settings. These are ReLU or tanh, one or two hidden layers, and 2, 8, 32 or 128 hidden units. Every setting uses an 80/20 train and holdout split, standardized features, a learning rate of 0.01, 200 epochs and a batch size of 32.

## Results at a glance

| Task | Dataset | Best holdout result | Best setting |
|---|---|---|---|
| Binary classification | `make_moons` | 99.0% accuracy | tanh, 2 layers, 128 units |
| Multi-class classification | `make_classification` | 85.5% accuracy | tanh, 2 layers, 32 units |
| Regression | `make_regression` | R² 0.443 | ReLU, 1 layer, 8 units |

Bigger networks helped a lot on the curved moons data. On the other two datasets, the models hit the limit set by the noise in the data, so extra units or layers did not help.

In [1]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score

ACTIVATIONS = {"relu": torch.relu, "tanh": torch.tanh}

configs = [(act, layers, hidden)
           for act in ["relu", "tanh"]
           for layers in [1, 2]
           for hidden in [2, 8, 32, 128]]


## 1. Design Choices

Before building the models, we decided on the cost function and output layer for each task, why the data needs to be standardized and how to spot overfitting.

Cost functions

For binary classification, we used binary cross-entropy. It gives a big penalty when the model is confident and wrong. For multi-class classification, we used categorical cross-entropy, which is the same idea for more than two classes. For regression, we used mean squared error, which measures the average distance between predictions and the true values.


Output layer

Only the output layer changes between tasks. For binary classification, it has 1 unit with a sigmoid, which gives a probability between 0 and 1. We predict class 1 when that probability is at least 0.5. For multi-class classification, it has one unit per class with a softmax, which gives a probability for each class. We predict the class with the highest probability. For regression, it has 1 unit with no activation, so the model can predict any number.


Standardizing the data

Standardizing puts all features on the same scale. Without it, features with bigger values would control the weight updates, and training would be slow or unstable. It also helps tanh, which stops learning well when inputs get too large. We fit the scaler on the training set only and then used it on the holdout set, so the holdout set stayed unseen. For regression, we also standardized the target because its values were in the hundreds, then converted the predictions back before measuring the error.


Detecting overfitting

We measured each model on both the training and holdout sets. If a model does much better on training data than on holdout data, it is learning noise instead of the real pattern. We also checked whether that gap grew as the models grew, which is a clear sign of overfitting.

## 2. Binary Classification

The data is two curved half circles from `make_moons` with 1,000 points and some noise. A straight line cannot separate them, so the model has to learn a curved boundary.

In [2]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
X_test_t = torch.tensor(X_test_s, dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_test_t = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

print(X_train_t.shape, y_train_t.shape)

torch.Size([800, 2]) torch.Size([800, 1])


In [4]:
class BinaryMLP:
    def __init__(self, n_inputs, n_hidden, n_layers=1, activation="relu"):
        self.activation = ACTIVATIONS[activation]
        self.n_layers = n_layers

        self.W1 = (torch.randn(n_inputs, n_hidden) / n_inputs ** 0.5).requires_grad_()
        self.b1 = torch.zeros(n_hidden, requires_grad=True)
        if n_layers == 2:
            self.W2 = (torch.randn(n_hidden, n_hidden) / n_hidden ** 0.5).requires_grad_()
            self.b2 = torch.zeros(n_hidden, requires_grad=True)
        # one output unit for the probability of class 1
        self.W_out = (torch.randn(n_hidden, 1) / n_hidden ** 0.5).requires_grad_()
        self.b_out = torch.zeros(1, requires_grad=True)

    def parameters(self):
        params = [self.W1, self.b1, self.W_out, self.b_out]
        if self.n_layers == 2:
            params += [self.W2, self.b2]
        return params

    def forward(self, X):
        h = self.activation(X @ self.W1 + self.b1)
        if self.n_layers == 2:
            h = self.activation(h @ self.W2 + self.b2)
        return torch.sigmoid(h @ self.W_out + self.b_out)

    def cost(self, y_hat, y):
        y_hat = torch.clamp(y_hat, 1e-7, 1 - 1e-7)
        return -torch.mean(y * torch.log(y_hat) + (1 - y) * torch.log(1 - y_hat))

    def fit(self, X, y, lr=0.01, epochs=200, batch_size=32):
        n = X.shape[0]
        self.history = []
        for epoch in range(epochs):
            perm = torch.randperm(n)
            for i in range(0, n, batch_size):
                idx = perm[i:i + batch_size]
                loss = self.cost(self.forward(X[idx]), y[idx])
                loss.backward()
                with torch.no_grad():
                    for p in self.parameters():
                        p -= lr * p.grad
                        p.grad.zero_()
            with torch.no_grad():
                self.history.append(self.cost(self.forward(X), y).item())
        return self

    def predict(self, X):
        with torch.no_grad():
            return (self.forward(X) >= 0.5).int().flatten()

In [5]:
rows = []
for act, layers, hidden in configs:
    torch.manual_seed(42)
    model = BinaryMLP(n_inputs=2, n_hidden=hidden, n_layers=layers, activation=act)
    model.fit(X_train_t, y_train_t, lr=0.01, epochs=200)
    rows.append({
        "activation": act, "hidden_layers": layers, "hidden_units": hidden,
        "train_acc": accuracy_score(y_train, model.predict(X_train_t).numpy()),
        "holdout_acc": accuracy_score(y_test, model.predict(X_test_t).numpy()),
    })

binary_results = pd.DataFrame(rows)
binary_results.round(3)

,activation,hidden_layers,hidden_units,train_acc,holdout_acc
0,relu,1,2,0.856,0.920
1,relu,1,8,0.914,0.950
2,relu,1,32,0.899,0.950
3,relu,1,128,0.910,0.950
4,relu,2,2,0.500,0.500
5,relu,2,8,0.959,0.965
6,relu,2,32,0.959,0.975
7,relu,2,128,0.962,0.975
8,tanh,1,2,0.856,0.915
9,tanh,1,8,0.899,0.935


Number of hidden units

The number of hidden units mattered a lot more here than in the multi-class task. With one ReLU layer, increasing from 2 to 8 units raised holdout accuracy from 92% to 95%, and increasing further to 32 or 128 units kept it at 95%. With one tanh layer, holdout accuracy went from 91.5% with 2 units to 97% with 128 units. We think this is because the two moons are curved and wrap around each other, so a straight line can't separate them. A network with only 2 hidden units can only bend the boundary a little, and more units let it follow each moon's curve.


One layer vs. two layers

Adding a second layer helped in almost every case. With 8 or more units, two ReLU layers reached 96.5% to 97.5% on the holdout set compared to 95% with one layer. Two tanh layers did even better, reaching 98% with 32 units and 99% with 128 units, which was the best result in the table. The one big failure was two ReLU layers with only 2 units, which got exactly 50% on both sets. That means the model predicted the same class for every point. We think both units ended up with negative inputs for every point, so ReLU set them to zero, and no gradient could flow back to fix them. This is called the dying ReLU problem, and it is much more likely when a layer only has 2 units.


ReLU vs. Tanh

Tanh did a little better overall. The best tanh model reached 99% on the holdout set, and the best ReLU model reached 97.5%. Tanh also handled the small network with two layers of 2 units fine at 91.5%, while ReLU collapsed. Tanh never outputs exactly zero for a whole range of inputs, so its units can't die the way ReLU units can. Its smooth curve may also fit the moons' round shape a little better than ReLU, which builds the boundary from straight pieces.


Overfitting

We did not see overfitting. Holdout accuracy was higher than training accuracy in every model, by up to 6 percentage points. Like the multi-class task, we think the 200 holdout points just happened to have fewer noisy points near the boundary than the training set. The largest model, two tanh layers with 128 units, also had the highest score on both sets, so the extra size did not hurt.

## 3. Multi-Class Classification

The data has three classes from `make_classification`. About 15% of the labels are randomly flipped (`flip_y=0.15`), so no model can score much higher than about 85%.

For this task, the output layer has one unit per class with a softmax activation, so the outputs are probabilities that add up to 1. The labels are one-hot encoded, the cost is categorical cross-entropy and the predicted class is the one with the highest probability.

In [6]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_classes=3, 
                           n_clusters_per_class=1, 
                           n_features=2, n_informative=2, n_redundant=0, random_state=1234, flip_y=0.15)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

encoder = OneHotEncoder(sparse_output=False)
y_train_oh = encoder.fit_transform(y_train.reshape(-1, 1))
y_test_oh = encoder.transform(y_test.reshape(-1, 1))

X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
X_test_t = torch.tensor(X_test_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train_oh, dtype=torch.float32)
y_test_t = torch.tensor(y_test_oh, dtype=torch.float32)

n_classes = y_train_oh.shape[1]

print(X_train_t.shape, y_train_t.shape)

torch.Size([800, 2]) torch.Size([800, 3])


In [8]:
class MultiClassMLP:
    def __init__(self, n_inputs, n_hidden, n_classes, n_layers=1, activation="relu"):
        self.activation = ACTIVATIONS[activation]
        self.n_layers = n_layers

        self.W1 = (torch.randn(n_inputs, n_hidden) / n_inputs ** 0.5).requires_grad_()
        self.b1 = torch.zeros(n_hidden, requires_grad=True)
        if n_layers == 2:
            self.W2 = (torch.randn(n_hidden, n_hidden) / n_hidden ** 0.5).requires_grad_()
            self.b2 = torch.zeros(n_hidden, requires_grad=True)
        # one output unit per class
        self.W_out = (torch.randn(n_hidden, n_classes) / n_hidden ** 0.5).requires_grad_()
        self.b_out = torch.zeros(n_classes, requires_grad=True)

    def parameters(self):
        params = [self.W1, self.b1, self.W_out, self.b_out]
        if self.n_layers == 2:
            params += [self.W2, self.b2]
        return params

    def forward(self, X):
        h = self.activation(X @ self.W1 + self.b1)
        if self.n_layers == 2:
            h = self.activation(h @ self.W2 + self.b2)
        return F.softmax(h @ self.W_out + self.b_out, dim=1)

    def cost(self, y_hat, y):
        # categorical cross-entropy, averaged over observations
        y_hat = torch.clamp(y_hat, 1e-7, 1.0)
        return -torch.mean(torch.sum(y * torch.log(y_hat), dim=1))

    def fit(self, X, y, lr=0.01, epochs=200, batch_size=32):
        n = X.shape[0]
        self.history = []
        for epoch in range(epochs):
            perm = torch.randperm(n)
            for i in range(0, n, batch_size):
                idx = perm[i:i + batch_size]
                loss = self.cost(self.forward(X[idx]), y[idx])
                loss.backward()
                with torch.no_grad():
                    for p in self.parameters():
                        p -= lr * p.grad
                        p.grad.zero_()
            with torch.no_grad():
                self.history.append(self.cost(self.forward(X), y).item())
        return self

    def predict(self, X):
        with torch.no_grad():
            return torch.argmax(self.forward(X), dim=1)   # class with highest probability


In [9]:
rows = []
for act, layers, hidden in configs:
    torch.manual_seed(42)
    model = MultiClassMLP(n_inputs=2, n_hidden=hidden, n_classes=n_classes,
                          n_layers=layers, activation=act)
    model.fit(X_train_t, y_train_t, lr=0.01, epochs=200)
    rows.append({
        "activation": act, "hidden_layers": layers, "hidden_units": hidden,
        "train_acc": accuracy_score(y_train, model.predict(X_train_t).numpy()),
        "holdout_acc": accuracy_score(y_test, model.predict(X_test_t).numpy()),
    })

multiclass_results = pd.DataFrame(rows)
multiclass_results.round(3)

,activation,hidden_layers,hidden_units,train_acc,holdout_acc
0,relu,1,2,0.785,0.840
1,relu,1,8,0.802,0.840
2,relu,1,32,0.804,0.850
3,relu,1,128,0.804,0.850
4,relu,2,2,0.775,0.815
5,relu,2,8,0.801,0.840
6,relu,2,32,0.804,0.845
7,relu,2,128,0.800,0.835
8,tanh,1,2,0.794,0.845
9,tanh,1,8,0.792,0.835


Number of hidden units

The number of hidden units barely changed anything. Every model landed between 77.5% and 80.4% on the training set and between 81% and 85.5% on the holdout set. With one ReLU layer, increasing from 2 units to 32 units raised holdout accuracy from 84% to 85%, and going up to 128 units added nothing. With one tanh layer, the 128-unit model was actually the worst one at 81%. We think this happened because the three classes are already close to separable with straight lines, so even a tiny network can draw decent boundaries. The bigger limit is the data itself. The dataset uses flip_y=0.15, which randomly switches about 15% of the labels, so around 85% is the best any model can really do. Our models were already hitting that ceiling.


One layer vs. two layers

Adding a second hidden layer didn't clearly improve performance. For ReLU, two layers did about the same or a little worse than one layer. For tanh, two layers with 8 or 32 units gave the best results in the whole table at 85% and 85.5%. That gain is only 1 or 2 percentage points, though, and with 200 holdout points, each half percent is just one observation. So we don't think the difference is meaningful. This problem is simple enough that the extra layer isn't needed.


ReLU vs. Tanh

ReLU and tanh performed almost the same. The best ReLU model reached 85% on the holdout set, and the best tanh model reached 85.5%. Both activations did poorly with two layers of only 2 units, which scored 81.5% for each. We think that network is too narrow to pass enough information from one layer to the next. Since the class boundaries are close to straight lines, the activation function's shape doesn't matter much here.


Overfitting

We did not see any overfitting. Holdout accuracy was higher than training accuracy for every single model. At first that looked strange, but it makes sense with this data. The 200 holdout points just happened to have a smaller share of flipped labels than the training set, so they were easier to get right.


## 4. Regression

The data has two features from `make_regression` with a lot of added noise (`noise=75`). The true relationship is a straight line.

In [10]:
from sklearn.datasets import make_regression
X, y = make_regression(n_samples=1000, n_features=2, n_informative=2, random_state=1234, noise=75)

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

y_scaler = StandardScaler()
y_train_s = y_scaler.fit_transform(y_train.reshape(-1, 1))

X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
X_test_t = torch.tensor(X_test_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train_s, dtype=torch.float32)

print(X_train_t.shape, y_train_t.shape)

torch.Size([800, 2]) torch.Size([800, 1])


In [12]:
class RegressionMLP:
    def __init__(self, n_inputs, n_hidden, n_layers=1, activation="relu", y_scaler=None):
        self.activation = ACTIVATIONS[activation]
        self.n_layers = n_layers
        self.y_scaler = y_scaler

        self.W1 = (torch.randn(n_inputs, n_hidden) / n_inputs ** 0.5).requires_grad_()
        self.b1 = torch.zeros(n_hidden, requires_grad=True)
        if n_layers == 2:
            self.W2 = (torch.randn(n_hidden, n_hidden) / n_hidden ** 0.5).requires_grad_()
            self.b2 = torch.zeros(n_hidden, requires_grad=True)
        # one output unit for the predicted value
        self.W_out = (torch.randn(n_hidden, 1) / n_hidden ** 0.5).requires_grad_()
        self.b_out = torch.zeros(1, requires_grad=True)

    def parameters(self):
        params = [self.W1, self.b1, self.W_out, self.b_out]
        if self.n_layers == 2:
            params += [self.W2, self.b2]
        return params

    def forward(self, X):
        h = self.activation(X @ self.W1 + self.b1)
        if self.n_layers == 2:
            h = self.activation(h @ self.W2 + self.b2)
        return h @ self.W_out + self.b_out

    def cost(self, y_hat, y):
        # mean squared error
        return torch.mean((y_hat - y) ** 2)

    def fit(self, X, y, lr=0.01, epochs=200, batch_size=32):
        n = X.shape[0]
        self.history = []
        for epoch in range(epochs):
            perm = torch.randperm(n)
            for i in range(0, n, batch_size):
                idx = perm[i:i + batch_size]
                loss = self.cost(self.forward(X[idx]), y[idx])
                loss.backward()
                with torch.no_grad():
                    for p in self.parameters():
                        p -= lr * p.grad
                        p.grad.zero_()
            with torch.no_grad():
                self.history.append(self.cost(self.forward(X), y).item())
        return self

    def predict(self, X):
        with torch.no_grad():
            out = self.forward(X).numpy()
        if self.y_scaler is not None:
            out = self.y_scaler.inverse_transform(out)
        return out.flatten()

In [13]:
rows = []
for act, layers, hidden in configs:
    torch.manual_seed(42)
    model = RegressionMLP(n_inputs=2, n_hidden=hidden, n_layers=layers,
                          activation=act, y_scaler=y_scaler)
    model.fit(X_train_t, y_train_t, lr=0.01, epochs=200)
    pred_train = model.predict(X_train_t)
    pred_test = model.predict(X_test_t)
    rows.append({
        "activation": act, "hidden_layers": layers, "hidden_units": hidden,
        "train_mse": mean_squared_error(y_train, pred_train),
        "holdout_mse": mean_squared_error(y_test, pred_test),
        "train_r2": r2_score(y_train, pred_train),
        "holdout_r2": r2_score(y_test, pred_test),
    })

regression_results = pd.DataFrame(rows)
regression_results.round(3)

,activation,hidden_layers,hidden_units,train_mse,holdout_mse,train_r2,holdout_r2
0,relu,1,2,5503.628,5469.546,0.504,0.443
1,relu,1,8,5398.286,5465.834,0.514,0.443
2,relu,1,32,5400.429,5473.267,0.514,0.442
3,relu,1,128,5388.068,5472.114,0.515,0.442
4,relu,2,2,5453.700,5512.484,0.509,0.438
5,relu,2,8,5362.770,5506.768,0.517,0.439
6,relu,2,32,5308.632,5535.171,0.522,0.436
7,relu,2,128,5288.655,5549.744,0.524,0.434
8,tanh,1,2,5535.339,5522.440,0.502,0.437
9,tanh,1,8,5405.436,5497.607,0.513,0.440


Number of hidden units

The number of hidden units had almost no effect on the holdout set. With one ReLU layer, holdout MSE stayed between about 5,466 and 5,473 no matter how many units we used, and holdout R² stayed at 0.44. Adding units did lower the training MSE a bit, from 5,504 with 2 units to 5,388 with 128 units, but that gain did not carry over to new data. We think this is because make_regression creates a linear relationship and then adds noise with a standard deviation of 75. The noise alone has a variance of 75 squared, or 5,625, so an MSE around 5,500 is close to the best any model can do. The leftover error is noise, not something more units can learn.


One layer vs. two layers

A second layer made things slightly worse on the holdout set. Two ReLU layers had the lowest training MSE in the table, down to 5,289 with 128 units, but their holdout MSE rose from 5,512 to 5,550 as we added units. The tanh models followed the same pattern, and two tanh layers with 32 units had the worst holdout MSE at 5,610. Since the true relationship is a straight line, the extra layer only gives the model more room to fit noise in the training data.


ReLU vs. Tanh

The two activations performed similarly, but ReLU was slightly better. The best ReLU model, one layer with 8 units, had a holdout MSE of 5,466 and R² of 0.443. The best tanh model, also one layer with 8 units, had a holdout MSE of 5,498 and an R² of 0.440. We think ReLU has a small edge because it is linear for positive inputs, so it can copy a straight-line relationship easily. Tanh is curved, so it has to work a little harder to act like a straight line.


Overfitting

This is the only task where we saw some overfitting. Training R² was higher than holdout R² for every model, around 0.51 compared to 0.44. Part of that gap shows up even in the smallest model, so some of it is just the holdout set being a little harder to predict. But the gap grows as the models get bigger. For two ReLU layers, it went from 0.071 with 2 units to 0.090 with 128 units. That shows the larger models were starting to fit noise in the training data. The effect is small because the data only has 2 features, and the noise is too random to memorize much.

## 5. Code Reuse with Inheritance

Each task has its own class, and much of the code repeats across the three. This section shows how inheritance could remove that repetition.

Our three classes repeat most of the same code. The setup, hidden layers, and training loop are almost the same across all three. Only four things change between tasks. These are the number of output units, the output activation, the cost function, and the predict method. Because we copied the shared code three times, any fix had to be made in three places.

With inheritance, we would put the shared code in one parent class called `BaseMLP`. Each task would be a small child class that adds only the parts that differ.

```python
class BaseMLP:
    def __init__(self, n_inputs, n_hidden, n_outputs, n_layers=1, activation="relu"):
        ...  # same weight setup for every task

    def hidden(self, X):
        h = self.activation(X @ self.W1 + self.b1)
        if self.n_layers == 2:
            h = self.activation(h @ self.W2 + self.b2)
        return h

    def forward(self, X):
        return self.output_activation(self.hidden(X) @ self.W_out + self.b_out)

    def fit(self, X, y, lr=0.01, epochs=200, batch_size=32):
        ...  # same training loop for every task


class BinaryMLP(BaseMLP):
    def __init__(self, n_inputs, n_hidden, **kwargs):
        super().__init__(n_inputs, n_hidden, n_outputs=1, **kwargs)

    def output_activation(self, z):
        return torch.sigmoid(z)

    def cost(self, y_hat, y):
        y_hat = torch.clamp(y_hat, 1e-7, 1 - 1e-7)
        return -torch.mean(y * torch.log(y_hat) + (1 - y) * torch.log(1 - y_hat))

    def predict(self, X):
        with torch.no_grad():
            return (self.forward(X) >= 0.5).int().flatten()
```

`MultiClassMLP` and `RegressionMLP` would work the same way, each with its own output activation, cost, and predict. Each child class would be about 15 lines instead of 50. If we wanted to change the training loop, we would only change it once in `BaseMLP`.


